# 5장 2강. 분석 결과 평가와 개선
## 실습문제


## 실습 목표

- 정답·원문과 비교하는 정확성과 반복 결과를 비교하는 일관성을 구분할 수 있다.
- 분류 정확도(Accuracy)를 계산하고, 오답이 발생한 상담을 확인할 수 있다.
- 정보 추출의 필드별 정확성과 요약의 사실 일치·핵심 정보 유지·불필요한 정보 생성을 평가할 수 있다.
- 동일한 평가 데이터로 프롬프트와 워크플로우의 가상 결과를 비교할 수 있다.
- 개선된 오류·남은 오류·새로운 오류를 구분하고 근거에 맞는 개선 방향을 제안할 수 있다.

## 실습 환경 / 데이터

- Python 3, Jupyter Notebook 또는 VS Code 노트북, pandas
- CSV 3개를 노트북과 같은 폴더에 두고, 다른 위치라면 `DATA_DIR`를 수정합니다.
- 이 파일의 준비 코드부터 차례대로 실행합니다. 이전 강의 노트북은 필요하지 않습니다.

| 파일 | 이번 실습에서의 역할 |
|---|---|
| 엘지유플러스_분류.csv | 상담 요건의 평가 정답·반복 결과·프롬프트 비교 |
| 엘지유플러스_요약.csv | 기존 요약 참고 및 가상 요약의 사실·핵심 정보 검토 |
| 엘지유플러스_질의응답.csv | 질문별 기대 답변을 원문에 대조하여 정보 추출 평가 |

원본 컬럼은 source_id, consulting_category, task_category, instruction, input, output입니다. source_id는 상담 ID, instruction은 지시·질문, input은 원문, output은 CSV에 이미 저장된 출력입니다. source_id 하나에 여러 작업이 있으므로 세부 작업과 질문도 지정합니다.

### 실습 범위와 자료 구분

교안의 명언·리뷰 사례를 상담 텍스트에 적용합니다. **필수 1·필수 2·과제 1**, 기초~중급 수준이며 과제도 필수와 같은 수준입니다.

- 원문·지시·기존 출력은 실제 CSV 데이터입니다. 기존 output을 무조건 정답으로 간주하지 않고 원문과 명시한 기준으로 확인합니다.
- 모든 새 분류·추출·요약 결과와 반복 실행 기록은 **작성자가 만든 가상 평가 자료**입니다. 실제 API를 실행하거나 모델 성능을 측정한 자료가 아닙니다.
- 같은 평가 데이터로 비교하는 계산과 해석을 연습합니다. 코드가 생성 결과의 의미를 자동으로 이해하고 평가하지는 않습니다.
- 원문 근거가 필요한 판단은 사람이 읽고 표에 작성합니다. 그 수동 판단을 파이썬으로 정리·집계합니다.
- API 호출·LLM 평가 모델·복잡한 평가 시스템·교안에 없는 평가 지표는 사용하지 않습니다.
- 가림 표시를 추측하여 복원하지 않습니다. 상담 내용은 해당 원문의 사실로 평가합니다.

| 교안 내용 | 실습 |
|---|---|
| 정확성과 일관성 | 필수 1: 분류 정확도와 3회 결과 비교 |
| 작업별 평가 기준 | 필수 2: 정보 추출의 필드와 요약의 내용 검토 |
| 프롬프트·워크플로우 비교 | 과제 1: 같은 문서·정답·기준으로 가상 결과 비교 |
| 오류 분석과 개선 검증 | 과제 1: 개선·악화·남은 오류와 추가 확인 계획 |


## 실습 준비

제공 코드를 실행하세요. 함수 구현 자체는 평가하지 않습니다.

1. **데이터 불러오기:** 세 파일을 별도 표로 읽습니다.
2. **분석 대상 컬럼 확인:** 원문·지시·기존 출력과 상담 ID를 확인합니다.
3. **결측치 및 자료형 확인:** 원본 컬럼의 자료형, 결측치와 빈 문자열을 확인합니다.
4. **기초 통계량과 분포 확인:** 고유 상담 수, 작업 유형별 행 수와 본문 길이를 확인합니다.

record_id는 파일 이름과 현재 행 위치로 만든 실습용 ID이며 원본의 영구 식별자는 아닙니다. 본문 글자 수는 토큰 수가 아닙니다. 파일의 작업 행 수가 다르다는 이유만으로 LLM의 결과 누락이라고 판단하지 않습니다.


In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path(".")
classification = pd.read_csv(DATA_DIR / "엘지유플러스_분류.csv")
summarization = pd.read_csv(DATA_DIR / "엘지유플러스_요약.csv")
qa = pd.read_csv(DATA_DIR / "엘지유플러스_질의응답.csv")
tables = {"분류": classification, "요약": summarization, "질의응답": qa}

for name, table in tables.items():
    print(f"\n[{name}] 원본 {table.shape[0]:,}행 × {table.shape[1]}열")
    print(pd.DataFrame({"자료형": table.dtypes.astype(str),
                        "결측치": table.isna().sum()}).to_string())
    print("고유 상담 ID 수:", table["source_id"].nunique())
    print("빈 본문 수:", table["input"].str.strip().eq("").sum())
    print("빈 지시 수:", table["instruction"].str.strip().eq("").sum())
    print("빈 출력 수:", table["output"].str.strip().eq("").sum())
    print("작업 유형별 행 수:")
    print(table["task_category"].value_counts().to_string())
    print("행별 본문 글자 수(토큰 수가 아님):")
    print(table["input"].str.len().agg(["min", "mean", "max"]).round(2).to_string())
    table["record_id"] = [f"{name}-{i}" for i in range(len(table))]

practice_ids = [40003, 40005]
for source_id in practice_ids:
    texts = []
    print(f"\n[연습 상담 {source_id}]")
    for name, table in tables.items():
        part = table[table["source_id"] == source_id]
        print(name, "행 수:", len(part))
        texts.extend(part["input"].tolist())
    print("세 파일에서 이 상담의 원문이 모두 동일:", pd.Series(texts).nunique() == 1)
    print("이 상담 원문의 글자 수:", len(texts[0]))

def get_record(table, source_id, task_category, instruction=None):
    selected = table[(table["source_id"] == source_id)
                     & (table["task_category"] == task_category)]
    if instruction is not None:
        selected = selected[selected["instruction"] == instruction]
    if len(selected) != 1:
        raise ValueError("한 행을 선택해야 합니다. ID·작업 유형·질문을 다시 확인하세요.")
    return selected.iloc[0]

def show_record(row, include_input=False):
    print("\nrecord_id:", row["record_id"], " / source_id:", row["source_id"])
    print("작업 유형:", row["task_category"])
    print("지시 또는 질문:", row["instruction"])
    if include_input:
        print("상담 원문:\n" + row["input"])
    print("CSV에 저장된 출력:", row["output"])


### 평가 데이터와 기준 준비 — 제공 코드

이번 **상담 요건** 평가 기준은 다음과 같습니다.

- 단일 요건 민원: 중심 요청·주제가 하나입니다. 그 요청의 절차·서류·처리 시점을 여러 번 질문해도 하나의 중심 요청이면 단일로 봅니다.
- 다수 요건 민원: 서로 구분되는 요청·주제가 둘 이상입니다. 상담 중 추가로 이어진 별도 주제도 포함합니다.

평가 상담은 40001·40003·40005·40006·40007입니다. 아래 정답은 이 기준으로 원문을 검토한 **실습용 평가 정답**이며, 데이터 전체에 대한 새 라벨을 만드는 작업은 아닙니다. 원본 CSV의 분류도 별도 컬럼으로 남깁니다.

| 상담 | 평가 정답 | 원문에서 확인할 근거 |
|---|---|---|
| 40001 | 다수 요건 민원 | 케이스·보험 코드·카드 할인·정지 횟수 문의 |
| 40003 | 다수 요건 민원 | 요금제 변경, 결합 할인, 멤버십 변경 관련 상담 |
| 40005 | 다수 요건 민원 | 통화 중 대기 문제와 추가 결합 할인 상담 |
| 40006 | 단일 요건 민원 | 휴대폰 해지라는 한 요청의 절차·서류·시점 문의 |
| 40007 | 다수 요건 민원 | 파손보험 가입·결합 할인·자동이체 카드 확인 |

제공 코드는 선택한 분류·요약 행의 원문 일치를 확인한 뒤 eval_documents를 만듭니다. `show_eval_document(ID)`로 원문을 읽을 수 있습니다. 전체 5개 중 다수 4개·단일 1개이므로 다양한 실무 상황을 대표하는 성능 평가셋이라고 보기는 어렵습니다.


In [ ]:
EVAL_IDS = [40001, 40003, 40005, 40006, 40007]
evaluation_labels = {40001: "다수 요건 민원", 40003: "다수 요건 민원", 40005: "다수 요건 민원",
                     40006: "단일 요건 민원", 40007: "다수 요건 민원"}
evaluation_reasons = {
    40001: "케이스·보험 코드·카드 할인·정지 횟수",
    40003: "요금제 변경·결합 할인·멤버십",
    40005: "통화 중 대기와 추가 결합 할인",
    40006: "휴대폰 해지 한 요청의 절차·서류·시점",
    40007: "파손보험 가입·결합 할인·자동이체 카드"
}
eval_rows = []
for sid in EVAL_IDS:
    cr = get_record(classification, sid, "상담 요건")
    sr = get_record(summarization, sid, "길이 제한 요약")
    if cr["input"] != sr["input"]:
        raise ValueError("분류와 요약의 원문을 확인하세요.")
    eval_rows.append({"document_id": sid, "text": cr["input"],
                      "class_record_id": cr["record_id"], "summary_record_id": sr["record_id"],
                      "CSV_분류": cr["output"], "CSV_요약": sr["output"],
                      "정답": evaluation_labels[sid], "정답_근거": evaluation_reasons[sid]})
eval_documents = pd.DataFrame(eval_rows)
print(eval_documents[["document_id", "CSV_분류", "정답", "정답_근거"]].to_string(index=False))
print("\n평가 범주 분포")
print(eval_documents["정답"].value_counts().to_string())

def show_eval_document(sid):
    selected = eval_documents[eval_documents["document_id"] == sid]
    if len(selected) != 1:
        raise ValueError("한 문서를 선택해야 합니다.")
    row = selected.iloc[0]
    print("\n상담 ID:", sid)
    print(row["text"])
    print("기존 요약:", row["CSV_요약"])


---
## 필수 1. 분류 정확성과 반복 결과의 일관성

### 문제 1-1. 정답 비교와 반복 결과 비교를 나누어 평가하기

#### 문제 설명

평가 상담 5개를 같은 입력·프롬프트·모델·설정으로 세 번 분석했다고 가정한 자료입니다. 이 조건은 **가상 실험의 가정**이며 실제 모델을 실행하지 않습니다.

| 상담 ID | 1회차 | 2회차 | 3회차 |
|---|---|---|---|
| 40001 | 다수 요건 민원 | 다수 요건 민원 | 다수 요건 민원 |
| 40003 | 단일 요건 민원 | 단일 요건 민원 | 단일 요건 민원 |
| 40005 | 다수 요건 민원 | 단일 요건 민원 | 다수 요건 민원 |
| 40006 | 단일 요건 민원 | 단일 요건 민원 | 단일 요건 민원 |
| 40007 | 다수 요건 민원 | 다수 요건 민원 | 다수 요건 민원 |

분류 정확도는 **정답과 일치한 문서 수 ÷ 전체 평가 문서 수**로 계산합니다. 일관성은 이번에 관찰한 세 값이 모두 같은지 확인합니다. 별도 복잡한 일관성 지표는 만들지 않습니다.

#### 요구사항

1. show_eval_document()로 다섯 원문과 정답 근거를 확인하세요.
2. 제공된 repeat_predictions를 document_id로 eval_documents에 연결하여 repeat_results를 만드세요.
3. 각 회차의 정답 여부를 비교하고 평가 문서 수·정답 건수·Accuracy·백분율을 담은 accuracy_table을 만드세요. 분모는 매 회차 5입니다.
4. 각 문서에 `3회_동일`과 `3회_모두정답`을 추가하세요. 전자는 세 출력끼리 비교하고, 후자는 각 출력을 정답과 비교합니다.
5. 한 번이라도 오답인 문서를 wrong_cases로 모으고, 왜 오답인지 원문 기준으로 적으세요.

**보조 구문:** `표.merge(..., on="document_id", validate="one_to_one")`, `표[회차] == 표["정답"]`, `참거짓.sum()`, `&`(두 조건 모두), `표[조건]`

#### 해석 질문

**Q1.** 회차별 정확도와 정답 건수는 얼마인가요? 3회 결과 15개를 한꺼번에 세는 것과 회차별 평가가 같은 의미인가요?

**Q2.** 40003과 40005는 정확성·일관성 측면에서 각각 어떤 사례인가요?

**Q3.** 세 번 같은 결과이면 앞으로도 항상 같거나 반드시 정답이라고 말할 수 있나요?

**Q4.** 전체 정확도 외에 오답 문서와 범주 분포를 확인해야 하는 이유는 무엇인가요? 이 5개만으로 실제 성능을 일반화할 수 있나요?

#### 제출 결과

- repeat_results, accuracy_table, wrong_cases
- 원문에 근거한 오답 설명과 Q1~Q4 답변


### 제공 자료

아래 셀을 실행한 뒤 풀이 코드를 작성하세요.


In [ ]:
# 제공 자료: 실제 반복 실행 로그가 아닌 가상 결과입니다.
repeat_predictions = pd.DataFrame({
    "document_id": [40001, 40003, 40005, 40006, 40007],
    "1회차": ["다수 요건 민원", "단일 요건 민원", "다수 요건 민원", "단일 요건 민원", "다수 요건 민원"],
    "2회차": ["다수 요건 민원", "단일 요건 민원", "단일 요건 민원", "단일 요건 민원", "다수 요건 민원"],
    "3회차": ["다수 요건 민원", "단일 요건 민원", "다수 요건 민원", "단일 요건 민원", "다수 요건 민원"]
})


In [ ]:
# 문제 1 풀이 코드를 작성하세요.


### 해석 답안 작성

- Q1.
- Q2.
- Q3.
- Q4.


---
## 필수 2. 정보 추출과 요약에 맞는 평가 기준 적용

### 문제 2-1. 표현 차이·잘못된 값·누락·원문 밖 정보 구분

#### 문제 설명

**정보 추출:** 상담 40003의 질의응답 3개를 이용해 결합 할인 수혜자, 초과 사용분 과금 여부, 기존 결합 할인 유지 여부를 확인합니다. 가상 결과는 각각 `남편분`, `추가 요금이 발생함`, 빈 문자열입니다.

먼저 평가 기준을 정합니다. 필수 항목에 비어 있지 않은 답이 있어야 하고, 같은 의미의 표현은 맞는 답으로 인정합니다. `남편`과 `남편분`은 이번 질문에서 같은 대상을 가리키므로 의미상 일치로 봅니다. 코드의 문자열 비교와 사람이 판단하는 의미 비교를 구분하세요.

**요약:** 상담 40005의 세 가상 요약을 평가합니다. 핵심 정보는 ① 통화 중 대기 서비스 재설정과 정상 확인, ② 추가 할인은 즉시 진행하지 않고 추후 상담 연락에 동의한 점입니다. 다음 세 기준을 각각 판단합니다.

| 기준 | True의 뜻 |
|---|---|
| 사실_일치 | 요약에 적힌 사실이 원문과 일치함 |
| 핵심_유지 | 위에서 지정한 핵심 정보 두 가지를 모두 유지함 |
| 원문밖_정보 | 원문에 없는 내용을 추가함. 이 열은 True이면 문제 있음 |

핵심 누락과 사실 왜곡은 구분합니다. 어떤 사실을 쓰지 않은 경우는 핵심_유지에서, 반대로 바꾸어 쓴 경우는 사실_일치에서 확인합니다. 한 요약에 여러 오류가 함께 있을 수 있습니다.

#### 요구사항

1. 제공 셀을 실행하여 질문·기존 답변·가상 추출 결과와 요약 A~C를 확인하세요. 원문은 필수 1에서 확인했으며 필요하면 다시 출력합니다.
2. 추출 결과에 `문자열_일치`, `항목_존재`, `의미_일치`, `판단_근거`를 추가하여 extraction_review를 만드세요. 앞 두 항목은 코드로 계산하고, 뒤 두 항목은 사람이 원문을 읽고 입력합니다.
3. 추출 결과의 정확한 필드 수와 전체 필드 수를 확인하세요. 표현 차이를 모두 오답으로 세거나 빈 값을 부정 답변으로 해석하지 않습니다.
4. 요약마다 세 기준과 근거를 적은 summary_review를 만드세요. CSV 요약과 문장이 완전히 같아야 한다는 기준을 사용하지 않습니다.
5. 발견한 정보 추출 오류·누락·요약 오류·환각을 구분하고 각 문제에 맞는 개선 방향을 적으세요.

#### 해석 질문

**Q1.** 문자열이 다르면 항상 추출 오답인가요? 남편분과 빈 문자열을 각각 어떻게 평가해야 하나요?

**Q2.** 요약 A와 C는 어떤 점에서 다르며, 요약 B에는 어떤 오류가 함께 있나요?

**Q3.** 자연스러운 문장이나 CSV 요약과의 문자열 일치만으로 요약 품질을 판단하면 안 되는 이유는 무엇인가요?

**Q4.** 원문에 없는 원인 생성, 핵심 누락, 출력 형식 오류에는 각각 어떤 개선이 필요한가요? 스키마만 수정하면 내용 오류도 해결되나요?

#### 제출 결과

- extraction_review, summary_review
- 필드별·요약별 판단 근거와 개선 방향
- Q1~Q4 답변


### 제공 자료

아래 셀을 실행한 뒤 풀이 코드를 작성하세요.


In [ ]:
extraction_specs = [
    ("결합 할인 수혜자", "의문사형", "고객의 결합 할인은 누구 앞으로 되어 있나요?", "남편분"),
    ("초과 사용분 과금", "의문사형", "요금제 변경 시 초과 사용분은 어떻게 처리되나요?", "추가 요금이 발생함"),
    ("기존 결합 할인 유지", "예/아니요형", "고객의 요금제 변경 시 기존 결합 할인은 유지되나요?", ""),
]
extraction_rows=[]
for field, task, question, predicted in extraction_specs:
    qr=get_record(qa,40003,task,question)
    extraction_rows.append({"source_id":40003,"record_id":qr["record_id"],"필드":field,
                           "질문":qr["instruction"],"원문":qr["input"],"기존_answer":qr["output"],
                           "가상_추출":predicted})
extraction_data=pd.DataFrame(extraction_rows)
print(extraction_data[["필드","질문","기존_answer","가상_추출"]].to_string(index=False))

# 작성자가 만든 가상 요약이며, 일부는 의도적으로 부정확합니다.
summary_candidates=pd.DataFrame({
    "예시":["A","B","C"],
    "source_id":[40005,40005,40005],
    "가상_요약":[
        "통화 중 대기 서비스를 재설정하고 시험 통화로 정상 작동을 확인했다. 추가 결합 할인은 즉시 연결하지 않고 추후 상담 연락을 받기로 했다.",
        "단말기 고장으로 통화 중 대기가 작동하지 않아 고객이 새 휴대폰을 구매했다. 고객은 추가 결합 할인 가입도 즉시 완료했다.",
        "통화 중 대기 서비스를 재설정한 후 시험 통화로 정상 작동을 확인했다."
    ]
})
print("\n기존 요약:", get_record(summarization,40005,"길이 제한 요약")["output"])
print(summary_candidates.to_string(index=False))


In [ ]:
# 문제 2 풀이 코드를 작성하세요.


### 해석 답안 작성

- Q1.
- Q2.
- Q3.
- Q4.


---
## 과제 1. 동일한 평가 데이터로 방법 비교와 개선 검증

### 문제 3-1. 개선된 사례·새 오류·남은 오류를 함께 확인하기

#### 문제 설명

같은 평가 상담 5개에 두 프롬프트를 적용했다고 가정합니다.

- **변경 전 A, Zero-shot:** 공통 분류 기준만 제공합니다.
- **변경 후 B, Few-shot:** 같은 기준에 별도 상담 40002·40011의 정답 예시를 추가합니다.

두 예시는 평가 상담 5개와 분리합니다. 원문을 확인하면 40002는 명의변경이라는 한 요청의 절차·서류·변경 시 혜택·방문 안내, 40011은 영화 월정액 해지·요금제 검토·멤버십 신청을 다룹니다. 예시 정답은 각각 단일·다수 요건 민원입니다.

입력·평가 정답·모델·생성 설정·출력 조건은 같고 예시 추가만 달라졌다고 **가정한 비교 자료**입니다. 필수 1의 반복 평가와 별도인 가상 실험이며, 실제 API를 호출한 결과가 아닙니다.

| 상담 | 변경 전 A | 변경 후 B |
|---|---|---|
| 40001 | 다수 요건 민원 | 다수 요건 민원 |
| 40003 | 단일 요건 민원 | 다수 요건 민원 |
| 40005 | 단일 요건 민원 | 다수 요건 민원 |
| 40006 | 단일 요건 민원 | 다수 요건 민원 |
| 40007 | 단일 요건 민원 | 단일 요건 민원 |

추가로 **단일 호출과 순차 처리**가 같은 문서에서 분류·요약을 수행한 별도 가상 결과를 제공합니다. 분류는 정답 비교로, 요약은 원문과의 사실 일치로 평가합니다. 호출 수는 정상적으로 처리되는 설계의 문서당 예상 횟수이며 실제 과금 기록이 아닙니다.

#### 요구사항

1. 제공된 example_table과 eval_documents의 ID 및 원문이 겹치지 않는지 확인하세요. 평가 정답을 예시로 넣지 않습니다.
2. 상담 40006에 대한 zero_prompt와 few_prompt를 구성하세요. 공통 기준·대상 원문은 동일하게 두고, Few-shot에만 두 예시의 원문·정답을 추가합니다. 프롬프트 문자열을 변수에 보관하고 구성 차이를 확인합니다.
3. 제공된 comparison_predictions를 ID로 연결하여 prompt_comparison을 만들고, 두 방법의 정답 건수와 Accuracy를 계산하세요.
4. 각 문서를 `정답 유지`, `개선`, `새 오류`, `오답 유지`로 구분하세요. 두 방법의 전체 정확도만 비교하지 않습니다.
5. 40006의 새 오류와 40007의 남은 오류에 대해 원문 근거, 가능한 원인, 추가 개선안을 작성하세요. 원인은 확정 사실이 아니라 검토할 가설로 적습니다. 작성한 개선안을 실제로 실행한 것처럼 표현하지 않습니다.
6. 제공된 workflow_outputs의 두 방식에 대해 분류 정답 건수·Accuracy, 사람이 판단한 요약 사실 일치 건수, 문서당·전체 예상 호출 수를 workflow_comparison으로 정리하세요. 제공된 요약을 읽어 수동 판단 근거도 확인합니다.
7. 두 비교를 바탕으로 어느 방식을 우선 검토할지 조건을 붙여 설명하고, 반복 실행·다른 데이터·처리 시간과 사용량에서 더 확인할 사항을 적으세요.

**판정 기준**

| 변경 전 정답 | 변경 후 정답 | 변화 |
|---|---|---|
| True | True | 정답 유지 |
| False | True | 개선 |
| True | False | 새 오류 |
| False | False | 오답 유지 |

**보조 구문:** `set(A) & set(B)`는 공통 값을 찾습니다. `예측 == 정답`, `참거짓.sum()`, `정답건수 / 평가문서수`, `pd.DataFrame(행_목록)`을 사용하세요.

#### 해석 질문

**Q1.** 두 프롬프트의 정확도는 얼마이며, 개선·새 오류·오답 유지 문서는 각각 무엇인가요? 변경 후 방식이 항상 좋다고 결론 내릴 수 있나요?

**Q2.** Few-shot 예시와 평가 데이터를 분리하는 이유는 무엇인가요? 서로 다른 문서로 두 방식을 평가하면 어떤 문제가 있나요?

**Q3.** 단일 호출과 순차 처리 비교에서 분류와 요약 결과는 각각 어떻게 달랐나요? 호출이 두 배면 실제 시간·비용도 정확히 두 배인가요?

**Q4.** 추가 개선안을 어떻게 검증해야 하나요? 같은 평가 정답에 맞춰 계속 수정하는 경우와, 출력 누락·중간 분류 오류가 발생한 경우를 함께 설명하세요.

#### 제출 결과

- 예시·평가 분리 확인, 두 프롬프트 문자열
- prompt_comparison 및 정확도·변화 유형 집계
- 오류별 개선안, workflow_comparison, 추가 검증 계획
- Q1~Q4 답변


### 제공 자료

아래 셀을 실행한 뒤 풀이 코드를 작성하세요.


In [ ]:
# 제공 자료: 평가셋 밖의 실제 상담을 프롬프트 예시로 준비합니다.
example_rows=[]
for sid, expected, reason in [
    (40002,"단일 요건 민원","명의변경 한 요청의 절차·서류·변경 영향·방문 안내"),
    (40011,"다수 요건 민원","영화 월정액 해지·요금제 검토·멤버십 신청")
]:
    r=get_record(classification,sid,"상담 요건")
    example_rows.append({"document_id":sid,"text":r["input"],"CSV_분류":r["output"],
                         "예시정답":expected,"판단근거":reason})
example_table=pd.DataFrame(example_rows)
BASE_INSTRUCTION=(
    "상담 요건을 단일 요건 민원 또는 다수 요건 민원 중 하나로 분류하세요. "
    "중심 요청·주제가 하나면 단일, 구분되는 요청·주제가 둘 이상이면 다수입니다. "
    "한 요청의 절차·서류·시점을 여러 번 질문한 것만으로 다수로 보지 마세요. "
    "상담 중 추가로 이어진 별도 주제도 포함하여 판단하세요. 범주명만 출력하세요."
)
comparison_predictions=pd.DataFrame({
    "document_id":[40001,40003,40005,40006,40007],
    "Zero-shot":["다수 요건 민원","단일 요건 민원","단일 요건 민원","단일 요건 민원","단일 요건 민원"],
    "Few-shot":["다수 요건 민원","다수 요건 민원","다수 요건 민원","다수 요건 민원","단일 요건 민원"]
})

# 별도 가상 워크플로우 비교입니다. 같은 입력·정답·평가 기준을 사용한다고 가정합니다.
# 두 방식의 요약은 비교를 단순화하려고 같은 문장으로 제공했습니다.
workflow_outputs=pd.DataFrame({
    "document_id":[40001,40003,40005,40006,40007],
    "단일_분류":["다수 요건 민원","다수 요건 민원","다수 요건 민원","다수 요건 민원","단일 요건 민원"],
    "순차_분류":["다수 요건 민원","다수 요건 민원","다수 요건 민원","단일 요건 민원","단일 요건 민원"],
    "공통_가상요약":[
        "케이스, 보험 코드, 카드 할인과 휴대폰 정지 횟수에 관한 문의와 안내가 있었다.",
        "요금제 변경 후 기존 결합 할인은 해지되었고 VIP 등급은 그대로 유지되었다.",
        "통화 중 대기 서비스를 재설정해 정상 작동을 확인했으며 추가 할인은 추후 상담 연락을 받기로 했다.",
        "휴대폰 해지에 필요한 서류와 팩스 또는 매장 방문 절차, 처리 시점에 관한 안내가 있었다.",
        "파손보험 가입과 결합 할인을 위한 방문 안내를 받고 등록된 자동이체 카드를 확인했다."
    ],
    # 사람이 원문과 비교한 판정입니다. 자동 평가 모델의 출력이 아닙니다.
    "요약_사실일치":[True,False,True,True,True]
})
print("Few-shot 예시")
print(example_table[["document_id","예시정답","판단근거"]].to_string(index=False))
print("\n가상 워크플로우 요약과 사람의 사실 일치 판정")
print(workflow_outputs[["document_id","공통_가상요약","요약_사실일치"]].to_string(index=False))


In [ ]:
# 문제 3 풀이 코드를 작성하세요.


### 해석 답안 작성

- Q1.
- Q2.
- Q3.
- Q4.


---
## 실습 마무리

아래 질문에 각각 2~3문장으로 답하세요.

1. **어떤 분석 문제가 있었는가?** 반복되는 오답, 출력 변동, 잘못된 추출·누락·환각과 개선 후의 새 오류를 정리하세요.
2. **어떤 방법을 적용했는가?** 분류 Accuracy, 필드별 의미 검토, 요약 기준, 동일 데이터 비교와 오류 분석을 설명하세요.
3. **무엇을 근거로 결론을 내렸는가?** 검증된 정답·원문·반복 기록·문서별 전후 비교를 구분하세요.

### 제출 전 확인

- 필수 1·필수 2·과제 1과 Q1~Q4를 모두 작성했나요?
- 정확성과 일관성, 문자열 비교와 의미 판단을 구분했나요?
- 사람이 판단한 평가값을 자동 채점이라고 표현하지 않았나요?
- 같은 문서와 같은 기준으로 비교하고 Few-shot 예시를 분리했나요?
- 개선된 오류뿐 아니라 새 오류와 남은 오류도 확인했나요?
- 가상 결과를 실제 모델 성능이나 검증된 개선 효과로 주장하지 않았나요?


### 마무리 답안

1.
2.
3.
